Prioritization of ligands based on expression values
================

In this notebook, we will extend the basic NicheNet analysis from [Perform NicheNet analysis: step-by-step analysis](steps.ipynb) by incorporating gene expression as part of the prioritization. While the original NicheNet only ranks ligands based on the ligand activity analysis, it is now also possible to prioritize ligands based on cell type and condition specificity of the ligand and receptor. We will again make use of mouse NICHE-seq data to explore intercellular communication in the T cell area in the inguinal lymph node before and 72 hours after lymphocytic choriomeningitis virus (LCMV) infection (Medaglia et al. 2017). 

Make sure you understand the different steps in a NicheNet analysis that are described in the basic vignette before proceeding with this tutorial.

# Prepare NicheNet analysis

Load required packages, read in the AnnData object with processed expression data of interacting cells and NicheNet’s ligand-target prior model, ligand-receptor network and weighted integrated networks.

In [1]:
from nichenetpy.prediction import LigandActivityPredictor
from nichenetpy.network import LigandReceptorNetwork, WeightedNetwork
from nichenetpy.wrappers import run_nichenet
from nichenetpy.utils import (
    read_matrix_from_csv,
    combine_by_key,
    combine_dicts
)
from nichenetpy.extraction import (
    get_expressed_genes,
    subset_ann,
    get_weighted_ligand_receptor_links,
    get_lfc_celltype
)
from nichenetpy.gene_symbol import mouse_alias_info
from nichenetpy.visualization import (
    prepare_ligand_target_visualization,
    prepare_ligand_receptor_visualization,
    heatmap_2d,
    heatmap_1d
)

from itertools import cycle, chain

import anndata
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

In [2]:
ann = anndata.io.read_h5ad("D:/Data/nichenetpy/annData/annData3531889.h5")
mouse_alias_info.alias_to_symbol(ann)
predictor = LigandActivityPredictor(*read_matrix_from_csv("D:/Data/nichenetpy/model/mouse/csv/ligand_target_matrix_mouse.csv"))
lr_network = LigandReceptorNetwork(filename="D:/Data/nichenetpy/model/mouse/csv/lr_network_mouse.csv")
lr_sig = WeightedNetwork(filename="D:/Data/nichenetpy/model/mouse/csv/lr_sig_mouse.csv")

# Perform the NicheNet analysis

We will use the sender-focused approach here.

In [3]:
sender_celltypes = ("CD4 T", "Treg", "Mono", "NK", "B", "DC")
res = run_nichenet(
    ann,
    predictor,
    lr_network,
    "CD8 T",
    "LCMV",
    "SS",
    sender_celltypes=sender_celltypes,
)
ligand_activities_sorted = res["ligand_activities_sorted_focused"]
best_upstream_ligands = res["best_upstream_ligands_focused"]
expressed_ligands = res["expressed_ligands"]
expressed_receptors = res["expressed_receptors"]

# Perform prioritization of ligand-receptor pairs

We will prioritize ligand-receptor pairs based on the following criteria
(with their corresponding weight names):

- Upregulation of the ligand in a sender cell type compared to other
  cell types: `de_ligand`
- Upregulation of the receptor in a receiver cell type: `de_receptor`
- Average expression of the ligand in the sender cell type:
  `exprs_ligand`
- Average expression of the receptor in the receiver cell type:
  `exprs_receptor`
- Condition-specificity of the ligand across all cell types:
  `ligand_condition_specificity`
- Condition-specificity of the receptor across all cell types:
  `receptor_condition_specificity`

This means that we will have to calculate:

- Differential expression of the ligand/receptor in a sender/receiver
  cell type
- The average expression of each ligand/receptor in each sender/receiver
  cell type
- Differential expression of the ligand/receptor between the two
  conditions

We provide a wrapper function `generate_info_tables` that will calculate
all these values for you. This function returns a list with three
dataframes:

- `sender_receiver_de`: differential expression of the ligand and
  receptor in the sender-receiver cell type pair. These were first
  calculated separately (i.e., DE of ligand in sender cell type, DE of
  receptor in receiver cell type based on FindAllMarkers) and then
  combined based on possible interactions from the lr_network.
- `sender_receiver_info`: the average expression of the ligand and
  receptor in sender-receiver cell type pairs
- `lr_condition_de`: differential expression of the ligand and receptor
  between the two conditions across all cell types.

Note that cell type specificity (i.e., the first four conditions) is
calculated only in the condition of interest.

The “scenario” argument can be either “case_control” or “one_condition”.
In “case_control” scenario, condition specificity is calculated.

In [4]:
lr_network_filtered = lr_network.subset_sep(expressed_ligands, expressed_receptors)

In [18]:
temp = ann.copy()
temp.var_names = temp.var["gene"]
sc.pp.log1p(temp, layer="data")
sc.tl.rank_genes_groups(
    temp,
    groupby="aggregate",
    groups=["LCMV"],
    reference="SS",
    method="wilcoxon",
    layer="data"
)

In [19]:
import pandas as pd
res = temp.uns["rank_genes_groups"]
pd.DataFrame({
    "gene": [e[0] for e in res["names"]],
    "score": [e[0] for e in res["scores"]],
    "pval": [e[0] for e in res["pvals"]],
    "pval_adj": [e[0] for e in res["pvals_adj"]],
    "lfc": [e[0] for e in res["logfoldchanges"]]
})

,gene,score,pval,pval_adj,lfc
0,Ifi27l2b,42.615490,0.000000e+00,0.000000e+00,3.888075
1,Irf7,40.150429,0.000000e+00,0.000000e+00,5.026062
2,Ly6a,38.226685,0.000000e+00,0.000000e+00,3.946182
3,Ifi27l2a,36.551476,1.689478e-292,5.719306e-289,3.953723
4,Stat1,33.107140,2.345703e-240,6.352632e-237,3.167132
...,...,...,...,...,...
13536,Gm7079,-18.351471,3.213006e-75,9.888025e-73,-1.273160
13537,Gm10925,-19.952774,1.417991e-88,4.683174e-86,-0.487612
13538,Tmsb4x,-23.004091,4.241957e-117,2.209244e-114,-0.712801
13539,Gm8730,-23.046385,1.598955e-117,8.660580e-115,-0.993375
